# Maximum likelihood in machine learning
Checks for the Note: least squares = Gaussian MLE, logistic regression = Bernoulli MLE, ridge = MAP with a
Gaussian prior, and the overfitting experiment.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy import stats, optimize
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.datasets import make_classification

# one seeded random generator for the whole notebook
rng = np.random.default_rng(0)

## 1. Gaussian noise: minimising the NLL gives the least squares line

In [ ]:
x = np.array([1, 2, 3, 4.0])
y = np.array([1.8, 4.3, 5.7, 8.2])
# NLL of the line y = w x with noise sigma = 1
nll = lambda w: -stats.norm(w * x, 1).logpdf(y).sum()
res = optimize.minimize_scalar(nll)
print("NLL minimum at w =", res.x.round(4), " closed form:", (x @ y / (x @ x)).round(4))
print("NLL(2) =", nll(2).round(2), "  MSE(2) =", np.mean((y - 2 * x) ** 2))
# scikit-learn's least squares, without intercept, agrees
print("LinearRegression:", LinearRegression(fit_intercept=False).fit(x[:, None], y).coef_.round(4))

The MLE of the noise variance is the mean squared residual.

In [ ]:
w_hat = x @ y / (x @ x)
# minimise over w and log(sigma) together
nll2 = lambda p: -stats.norm(p[0] * x, np.exp(p[1])).logpdf(y).sum()
p = optimize.minimize(nll2, x0=[1, 0]).x
print("numerical sigma^2:", np.exp(p[1]) ** 2, "  mean squared residual:", np.mean((y - w_hat * x) ** 2))

Laplace noise: minimising the Laplace NLL gives the same w as minimising the sum of absolute errors.

In [ ]:
nll_lap = lambda w: -stats.laplace(w * x, 1).logpdf(y).sum()
sae = lambda w: np.abs(y - w * x).sum()
w_grid = np.linspace(1.5, 2.5, 100001)
print("Laplace NLL minimum:", w_grid[np.argmin([nll_lap(w) for w in w_grid[::100]]) * 100].round(3),
      " sum of absolute errors minimum:", w_grid[np.argmin([sae(w) for w in w_grid[::100]]) * 100].round(3))

## 2. Bernoulli: logistic regression is the minimiser of the Bernoulli NLL

In [ ]:
X, t = make_classification(n_samples=200, n_features=2, n_redundant=0, random_state=0)
sigmoid = lambda z: 1 / (1 + np.exp(-z))


def bern_nll(theta):
    # predicted probability of class 1
    p = sigmoid(X @ theta[:2] + theta[2])
    # minus the log of the Bernoulli PMF p^t (1 - p)^(1 - t)
    return -np.sum(t * np.log(p) + (1 - t) * np.log(1 - p))


ours = optimize.minimize(bern_nll, np.zeros(3)).x
sk = LogisticRegression(C=np.inf).fit(X, t)
print("our NLL minimiser:  ", ours.round(3))
print("scikit-learn (C = inf, no penalty):", np.r_[sk.coef_[0], sk.intercept_].round(3))

The four rows of the log loss Note: NLL of model 1.

In [ ]:
labels, y_hat = np.array([1, 0, 1, 0]), np.array([0.7, 0.6, 0.4, 0.2])
pmf = y_hat ** labels * (1 - y_hat) ** (1 - labels)
print("Bernoulli PMF of each row:", pmf, "  NLL:", -np.log(pmf).sum().round(2))

## 3. MAP with a Gaussian prior is ridge regression

In [ ]:
sigma, b = 1.0, 0.5
lam = sigma ** 2 / b ** 2
# MAP: minimise NLL minus log prior
neg_post = lambda w: -stats.norm(w * x, sigma).logpdf(y).sum() - stats.norm(0, b).logpdf(w)
print("MAP by optimisation:", optimize.minimize_scalar(neg_post).x.round(4))
print("closed form sum(xy) / (sum(x^2) + lambda):", (x @ y / (x @ x + lam)).round(4))
# Ridge minimises sum of squared errors + alpha * w^2
print("Ridge(alpha = 4):", Ridge(alpha=lam, fit_intercept=False).fit(x[:, None], y).coef_.round(4))

## 4. Overfitting: maximum likelihood polynomials of degree 0 to 9
Same data as Figure 3 of the Note.

In [ ]:
r4 = np.random.default_rng(4)
truth = lambda v: -np.sin(v / 5) + np.cos(v)
xs = np.linspace(-4.5, 4.5, 10) + r4.uniform(-0.3, 0.3, 10)
ys = truth(xs) + r4.normal(0, 0.2, 10)
xt = np.linspace(-4.5, 4.5, 200)
yt = truth(xt) + r4.normal(0, 0.2, 200)
phi = lambda v, M: np.vander(v, M + 1, increasing=True)
rmse = lambda th, v, t: np.sqrt(np.mean((phi(v, len(th) - 1) @ th - t) ** 2))
for M in range(10):
    th = np.linalg.lstsq(phi(xs, M), ys, rcond=None)[0]
    print(f"degree {M}: train {rmse(th, xs, ys):.3f}  test {rmse(th, xt, yt):.3f}  largest |coef| {np.abs(th).max():.2f}")

MAP (ridge) for degree 9 at several prior widths b, with sigma = 0.2. Smaller b means a stronger pull to 0.

In [ ]:
for b9 in (1, 0.3, 0.1, 0.03):
    lam9 = 0.2 ** 2 / b9 ** 2
    th = np.linalg.solve(phi(xs, 9).T @ phi(xs, 9) + lam9 * np.eye(10), phi(xs, 9).T @ ys)
    print(f"b = {b9}: lambda = {lam9:.2f}, test RMSE {rmse(th, xt, yt):.3f}")